# 02 Target Definition

Este notebook define o target final para modelagem, divide as bases de treino e OOT, e salva os conjuntos de dados finais.


## 1. Imports e Configurações

Definimos o período de treino e OOT, além das configurações de aleatoriedade para a amostragem de teste e OOS.


In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path('..').resolve()))

from src.data_loader import load_historico_emprestimos, load_historico_parcelas
from src.population import build_active_population
from src.target import build_contract_target, build_population_target, choose_target_definition

RANDOM_STATE = 42
TRAIN_PERIODS = [f"{year}-{month:02d}" for year in range(2020, 2024 + 1) for month in range(1, 13) if not (year == 2024 and month > 5)]
OOT_PERIODS = [f"2024-{month:02d}" for month in range(7, 13)]

print("TRAIN_PERIODS desde", TRAIN_PERIODS[0], "até", TRAIN_PERIODS[-1])
print("OOT_PERIODS desde", OOT_PERIODS[0], "até", OOT_PERIODS[-1])


TRAIN_PERIODS desde 2020-01 até 2024-05
OOT_PERIODS desde 2024-07 até 2024-12


## 2. Carregamento das Bases

Carregamos a população ativa e os históricos usados tanto para a definição dos possíveis targets quanto para o target final.


In [2]:
population_active = build_active_population()
historico_emprestimos = load_historico_emprestimos()
historico_parcelas = load_historico_parcelas()

print("population_active", population_active.shape)
print("historico_emprestimos", historico_emprestimos.shape)
print("historico_parcelas", historico_parcelas.shape)


population_active (35328, 24)
historico_emprestimos (186890, 37)
historico_parcelas (1390978, 8)


## 3. Construção do Target

Aqui a ideia é simples:

- `ever_30`, `ever_60` e `ever_90` são flags construídas em nível de contrato.
- Para cada contrato, calcula-se o atraso máximo observado nas parcelas usando o histórico de parcelas:
    - compara-se `data_prevista_pagamento` com `data_real_pagamento`;
    - obtém-se o atraso em dias para cada parcela;
    - toma-se o maior atraso observado para o contrato (`max_delay`).
- A partir desse atraso máximo, define-se:
    - `ever_30 = True` se o contrato teve ao menos uma parcela com atraso >= 30 dias;
    - `ever_60 = True` se o contrato teve ao menos uma parcela com atraso >= 60 dias;
    - `ever_90 = True` se o contrato teve ao menos uma parcela com atraso >= 90 dias.
- Essas variáveis viram indicadores de inadimplência por severidade e são usadas para comparar possíveis definições de target.


### 3.1 Possíveis definições de target

Vamos comparar as flags `ever_30`, `ever_60` e `ever_90` para escolher a definição mais adequada de inadimplência.


In [3]:
contract_target = build_contract_target()
contract_target = contract_target.merge(
    historico_emprestimos[["id_contrato", "id_cliente"]].drop_duplicates(),
    on="id_contrato",
    how="left",
)
candidate_target = contract_target.copy()
candidate_target["target_30"] = candidate_target["ever_30"].astype(int)
candidate_target["target_60"] = candidate_target["ever_60"].astype(int)
candidate_target["target_90"] = candidate_target["ever_90"].astype(int)

summary = pd.DataFrame({
    "definition": ["ever_30", "ever_60", "ever_90"],
    "bad_rate": [
        candidate_target["target_30"].mean(),
        candidate_target["target_60"].mean(),
        candidate_target["target_90"].mean(),
    ],
    "bad_count": [
        candidate_target["target_30"].sum(),
        candidate_target["target_60"].sum(),
        candidate_target["target_90"].sum(),
    ],
    "contract_count": [
        len(candidate_target),
        len(candidate_target),
        len(candidate_target),
    ],
})
active_client_ids = set(population_active["id_cliente"].dropna().unique())
active_contract_target = candidate_target[candidate_target["id_cliente"].isin(active_client_ids)].copy()
summary["bad_rate_active_clients"] = [
    active_contract_target["target_30"].mean(),
    active_contract_target["target_60"].mean(),
    active_contract_target["target_90"].mean(),
]
summary["bad_count_active_clients"] = [
    active_contract_target["target_30"].sum(),
    active_contract_target["target_60"].sum(),
    active_contract_target["target_90"].sum(),
]
summary["contract_count_active_clients"] = [
    len(active_contract_target),
    len(active_contract_target),
    len(active_contract_target),
]
summary["bad_rate_active_clients"] = summary["bad_rate_active_clients"].round(4)
display(summary)


,definition,bad_rate,bad_count,contract_count,bad_rate_active_clients,bad_count_active_clients,contract_count_active_clients
0,ever_30,0.021765,2338,107419,0.0203,2114,103915
1,ever_60,0.009747,1047,107419,0.0091,945,103915
2,ever_90,0.007783,836,107419,0.0072,748,103915


**Escolha do target**

No contexto de mercado, a inadimplência costuma ser definida a partir de 60 dias porque:
- atrasos de até 30 dias muitas vezes refletem problemas operacionais temporários, ajustes de fluxo de caixa ou erros no processamento, e o cliente pode regularizar a dívida rapidamente;
- a partir de 60 dias, a probabilidade de recuperação espontânea cai bastante, indicando um problema de pagamento mais sério e persistente;
- 60 dias captura melhor o risco de crédito relevante, sem reduzir demais o número de casos positivos disponíveis para modelagem;
- essa janela é mais alinhada às práticas de bancos e varejistas, que usam prazos mais longos para diferenciar inadimplência real de atrasos passageiros.

Com base nos dados, a escolha do `ever_60` fica ainda mais consistente:
- `ever_30` tem uma taxa de inadimplência mais alta (~2,18%), mas inclui atrasos leves e transitórios, o que tende a gerar mais ruído no modelo;
- `ever_90` é um evento muito raro (~0,78%), com poucos casos positivos, o que prejudica a capacidade de treinamento e generalização do modelo;
- `ever_60` apresenta uma taxa intermediária (~0,97%) e volume de casos positivos suficiente para capturar risco relevante sem diluir o sinal.

Portanto, `ever_60` foi escolhido por equilibrar melhor o critério de mercado com a robustez dos dados: sinal mais estruturado que `ever_30` e maior representatividade que `ever_90`.


### 3.2 Target final com filtro na população ativa

Construímos o target final em nível de cliente utilizando a população ativa e os históricos de contratos e parcelas.


In [ ]:
population_target = build_population_target(population_active)
area_target = population_target[["id_cliente", "target"]].copy()

population_active = population_active.merge(area_target, on="id_cliente", how="left")
population_active["target"] = population_active["target"].fillna(0).astype(int)
population_active = population_active.sort_values(["safra_mes", "id_cliente"]).reset_index(drop=True)

display(population_active.head())
print("Target shape:", population_active.shape)


,id_cliente,data_solicitacao,dia_semana_solicitacao_submissao,hora_solicitacao_submissao,tipo_contrato_submissao,valor_credito_submissao,valor_bem_submissao,valor_parcela_submissao,sexo,data_nascimento,...,tipo_organizacao,nivel_educacao,estado_civil,tipo_moradia,possui_carro,possui_imovel,nota_regiao_cliente,nota_regiao_cliente_cidade,safra_mes,target
0,101294,2020-01-31,FRIDAY,15,Cash loans,164223.0,135000.0,18571.995,F,1980-07-14,...,Industry: type 11,Secondary / secondary special,Married,House / apartment,N,Y,2,2,2020-01,0
1,101684,2020-01-26,SUNDAY,14,Consumer loans,122773.5,115173.0,14851.035,F,1997-09-01,...,Business Entity Type 3,Incomplete higher,Married,House / apartment,N,Y,2,2,2020-01,0
2,128768,2020-01-31,FRIDAY,8,Consumer loans,29538.0,29155.5,2735.235,M,1970-05-09,...,Business Entity Type 2,Secondary / secondary special,Separated,House / apartment,N,N,2,2,2020-01,0
3,135393,2020-01-14,TUESDAY,10,Consumer loans,97456.5,116995.5,21927.735,F,1963-07-30,...,XNA,Secondary / secondary special,Married,House / apartment,N,N,3,2,2020-01,0
4,141246,2020-01-05,SUNDAY,19,Consumer loans,37620.0,45270.0,7670.700,M,1966-01-27,...,Industry: type 3,Higher education,Married,House / apartment,N,N,1,1,2020-01,0


Target shape: (35328, 25)


## 4. Divisão de Treino e OOT

Dividimos o dataset em treino e OOT com base na safra, e em seguida extraímos um conjunto de teste e OOS a partir do treino.


In [5]:
train_base = population_active[population_active["safra_mes"].isin(TRAIN_PERIODS)].copy()
oot_base = population_active[population_active["safra_mes"].isin(OOT_PERIODS)].copy()

train_oos = train_base.sample(frac=0.20, random_state=RANDOM_STATE)
train_test = train_base.drop(train_oos.index).copy()

train_base = train_base.reset_index(drop=True)
train_test = train_test.reset_index(drop=True)
train_oos = train_oos.reset_index(drop=True)
oot_base = oot_base.reset_index(drop=True)

print("Treino total:", train_base.shape)
print("Teste (parte do treino):", train_test.shape)
print("OOS (parte do treino):", train_oos.shape)
print("OOT:", oot_base.shape)


Treino total: (26940, 25)
Teste (parte do treino): (21552, 25)
OOS (parte do treino): (5388, 25)
OOT: (6794, 25)


### 4.1 Distribuição de target por base

Mostramos a distribuição total de `target=1` (bad) e a distribuição por safra em cada base.


### 4.2 Teste

A base `teste` corresponde a 80% aleatórios da base de treino. Ela é utilizada para avaliação dentro do universo de treino.

### 4.3 OOS

A base `OOS` corresponde a 20% aleatórios da base de treino. Ela é usada como validação fora de amostra dentro do período de treino.


In [6]:
def print_target_distribution(df, name):
    print(f"=== {name} ===")
    totals = df["target"].value_counts().rename(index={0: "good", 1: "bad"})
    print("Total de registros:")
    print(totals)
    print()
    print("Distribuição por safra:")
    display(
        df.groupby("safra_mes")["target"]
        .agg(total="count", bad_count="sum", bad_rate="mean")
        .sort_index()
    )
    print()

for name, df in [
    ("Treino", train_base),
    ("Teste", train_test),
    ("OOS", train_oos),
    ("OOT", oot_base),
]:
    print_target_distribution(df, name)


=== Treino ===
Total de registros:
target
good    26746
bad       194
Name: count, dtype: int64

Distribuição por safra:


,total,bad_count,bad_rate
safra_mes,,,
2020-01,57,0,0.000000
2020-02,84,0,0.000000
2020-03,90,0,0.000000
2020-04,91,0,0.000000
2020-05,102,0,0.000000
2020-06,70,0,0.000000
2020-07,80,0,0.000000
2020-08,79,0,0.000000
2020-09,79,0,0.000000



=== Teste ===
Total de registros:
target
good    21396
bad       156
Name: count, dtype: int64

Distribuição por safra:


,total,bad_count,bad_rate
safra_mes,,,
2020-01,47,0,0.000000
2020-02,71,0,0.000000
2020-03,73,0,0.000000
2020-04,75,0,0.000000
2020-05,80,0,0.000000
2020-06,50,0,0.000000
2020-07,67,0,0.000000
2020-08,67,0,0.000000
2020-09,63,0,0.000000



=== OOS ===
Total de registros:
target
good    5350
bad       38
Name: count, dtype: int64

Distribuição por safra:


,total,bad_count,bad_rate
safra_mes,,,
2020-01,10,0,0.000000
2020-02,13,0,0.000000
2020-03,17,0,0.000000
2020-04,16,0,0.000000
2020-05,22,0,0.000000
2020-06,20,0,0.000000
2020-07,13,0,0.000000
2020-08,12,0,0.000000
2020-09,16,0,0.000000



=== OOT ===
Total de registros:
target
good    6736
bad       58
Name: count, dtype: int64

Distribuição por safra:


,total,bad_count,bad_rate
safra_mes,,,
2024-07,1504,10,0.006649
2024-08,1539,13,0.008447
2024-09,1160,7,0.006034
2024-10,1085,12,0.011060
2024-11,821,8,0.009744
2024-12,685,8,0.011679


In [7]:
output_path = Path("..") / "data" / "processed"
output_path.mkdir(parents=True, exist_ok=True)
train_base.to_parquet(output_path / "population_treino.parquet", index=False)
train_test.to_parquet(output_path / "population_teste.parquet", index=False)
train_oos.to_parquet(output_path / "population_oos.parquet", index=False)
oot_base.to_parquet(output_path / "population_oot.parquet", index=False)
print("Salvas as 4 bases finais em:", output_path)


Salvas as 4 bases finais em: ..\data\processed
